# Football Player Detection using YOLOv11

## Dataset Inspection & Validation

This notebook implements a complete dataset inspection pipeline before training a YOLOv11 object detection model.

### Pipeline

- Install dependencies
- Download dataset
- Inspect dataset structure
- Validate annotations
- Explore dataset statistics
- Create training configuration
- Load pretrained YOLOv11 model

In [1]:
# ============================================================
# Install Dependencies
# ============================================================

!pip install -q \
autodistill \
autodistill-grounded-sam \
autodistill-yolov8 \
supervision==0.9.0

!pip install -q roboflow ultralytics

In [2]:
# ============================================================
# Imports
# ============================================================

import os
import cv2
import yaml
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from roboflow import Roboflow
from ultralytics import YOLO

In [3]:
# ============================================================
# Download Dataset
# ============================================================

rf = Roboflow(api_key="ROBOFLOW_API_KEY")

project = (
    rf.workspace("kam-s2czz")
      .project("football-players-detection-3zvbc-vmzwi")
)

version = project.version(1)

dataset = version.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to football-players-detection-1 in yolov11:: 100%|██████████| 749/749 [00:00<00:00, 4006.49it/s]


In [4]:
# ============================================================
# Dataset Utilities
# ============================================================

def get_dataset_path(dataset):
    """
    Return dataset root directory.
    """

    dataset_path = Path(dataset.location)

    print(f"Dataset location:\n{dataset_path}")

    return dataset_path


def print_folder_structure(root_path, max_depth=2):
    """
    Print dataset directory tree.
    """

    root_path = Path(root_path)

    for path, dirs, files in os.walk(root_path):

        depth = len(Path(path).relative_to(root_path).parts)

        if depth <= max_depth:

            indent = "    " * depth

            print(f"{indent}{Path(path).name}/")

            for file in sorted(files)[:5]:
                print(f"{indent}    {file}")

In [5]:
# ============================================================
# Dataset Structure Inspection
# ============================================================

dataset_path = get_dataset_path(dataset)

print_folder_structure(dataset_path)

Dataset location:
/content/football-players-detection-1
football-players-detection-1/
    README.dataset.txt
    README.roboflow.txt
    data.yaml
    test/
        labels/
            08fd33_3_6_png.rf.4fc86ec972877bd9ec9814648580c33a.txt
            08fd33_9_3_png.rf.e815b9d20d0b4e6bf542352ce4e22448.txt
            40cd38_7_6_png.rf.779169d30fbed6f8c52e968f7815e9dc.txt
            42ba34_1_5_png.rf.73d15cfe0aafd832e8c19e1f73860cb8.txt
            42ba34_5_5_png.rf.c71544ff796889d208f073e1cdb03d41.txt
        images/
            08fd33_3_6_png.rf.4fc86ec972877bd9ec9814648580c33a.jpg
            08fd33_9_3_png.rf.e815b9d20d0b4e6bf542352ce4e22448.jpg
            40cd38_7_6_png.rf.779169d30fbed6f8c52e968f7815e9dc.jpg
            42ba34_1_5_png.rf.73d15cfe0aafd832e8c19e1f73860cb8.jpg
            42ba34_5_5_png.rf.c71544ff796889d208f073e1cdb03d41.jpg
    valid/
        labels/
            08fd33_3_1_png.rf.374da235747436cd3f95466bf4e76d69.txt
            08fd33_3_3_png.rf.a1e81a32135d12ca3